# 🌌 The Anatomy of a Black Hole — Cloud GPU Renderer
### Hollywood-Grade 3D Cinematic Short Film (Blender `bpy` + Cycles GPU)

This notebook runs on **Google Colab** with free NVIDIA T4, V100, or A100 GPUs to render high-sample Cycles frames and compile the final cinematic film directly to your Google Drive.

## 1. Verify GPU Acceleration & Mount Google Drive

In [ ]:
# 1. Verify GPU is active
!nvidia-smi

# 2. Mount Google Drive so rendered frames save permanently to your cloud storage
from google.colab import drive
drive.mount('/content/drive')

# Create permanent output folder in your Google Drive
!mkdir -p /content/drive/MyDrive/blackhole_film/frames
print('✓ Google Drive connected and output folder ready!')

## 2. Install Portable Blender 4.2 LTS (Fast Linux Tarball)

In [ ]:
import os
# Install aria2 for ultra-fast multi-threaded download (~10 seconds)
!apt-get install -y -qq aria2

if not os.path.exists('/content/blender-4.2.0-linux-x64'):
    print('Downloading portable Blender 4.2 LTS...')
    !aria2c -x 8 -s 8 -k 1M -q https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz || wget -q https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz
    print('Extracting Blender...')
    !tar -xf blender-4.2.0-linux-x64.tar.xz

# Add Blender to PATH
os.environ['PATH'] = '/content/blender-4.2.0-linux-x64:' + os.environ['PATH']
!blender --version

## 3. Clone or Sync Project Code

In [ ]:
import os
if not os.path.exists('/content/bpy'):
    !git clone https://github.com/SHADOW-MHMD/blackhole-cinematic-bpy.git /content/bpy
else:
    %cd /content/bpy
    !git pull

%cd /content/bpy
print('✓ Project code synced successfully!')

## 4. Test Render (Single Beauty Frame Preview)

In [ ]:
# Render frame 2000 (Act 2: The Accretion Storm with Doppler beaming) at 128 samples
!blender -b -P build_scene.py -- --render-frame 2000 --output /content/test_preview.png --samples 128

# Display rendered beauty image directly in the notebook
from IPython.display import Image, display
display(Image('/content/test_preview.png', width=960))

## 5. Production Batch Render (Cycles GPU 256–512 Samples to Google Drive)

In [ ]:
# You can render by Act or custom frame ranges:
# --act 1 : The Approach (Frames 1 - 1440)
# --act 2 : The Accretion Storm (Frames 1441 - 3600)
# --act 3 : The Plunge to the Photon Sphere (Frames 3601 - 5400)
# --act 4 : The Singularity & Beyond (Frames 5401 - 6480)
# Note: Automatically skips any frames that are already saved in your Google Drive!

!blender -b -P colab/render_colab.py -- --act 1 --samples 256

## 6. Assemble Final Master Film (Frames + Audio via FFmpeg)

In [ ]:
# Combine rendered frames from Google Drive with the master audio track
!chmod +x assemble_film.sh
!./assemble_film.sh /content/drive/MyDrive/blackhole_film/frames audio/ambient_score_foley.wav /content/drive/MyDrive/blackhole_film/The_Anatomy_of_a_Black_Hole_Master.mp4

print('✓ Master film saved to your Google Drive: /content/drive/MyDrive/blackhole_film/The_Anatomy_of_a_Black_Hole_Master.mp4')